`[1]`

# CNN de favorabilidade de fosfato — predição (mapa) no Colab

Gera o mapa de p de uma rodada treinada no `cnn_colab.ipynb`, por janela
deslizante sobre os pedaços de canais (`inferir_cnn.py --tiles`). Sem Earth
Engine até a célula 9 (opcional, subir ao GEE).

| Etapa | Onde | Células |
|---|---|---|
| exportar os canais da área (GeoTIFF por célula) | máquina local: `python exportar_tiles_drive.py --modelo A` (e `B1`) → Drive `tiles_cnn_gee/` | — |
| cortar em 6 × 6 pedaços com borda (uma vez por modelo) | aqui: `cortar_tiles_drive.py` | 5 |
| escolher a rodada (padrão: a melhor do ranking do `cnn_colab`) | aqui | 6 |
| mapa por janela deslizante | aqui: `inferir_cnn.py --tiles` | 7 |
| ver o mapa e o p nos pontos de fosfato | aqui | 8 |
| subir ao GEE (opcional) | aqui: `subir_predicao_gee.py` | 9 |

No Drive, `prospectra/models_scripts/` precisa de: `inferir_cnn.py`,
`treinar_cnn.py`, `exportar_patches.py`, `baixar_tiles.py`,
`cortar_tiles_drive.py` (e `subir_predicao_gee.py` para a célula 9).

**Antes:** *Ambiente de execução → Alterar o tipo → GPU*.

In [ ]:
# [2] Drive, código do projeto e pastas
from google.colab import drive
drive.mount('/content/drive')

import os, sys, json
import pandas as pd
from pathlib import Path

SCRIPTS_DIR = '/content/drive/MyDrive/prospectra/models_scripts'  #@param {type:"string"}
RUNS_DIR = '/content/drive/MyDrive/prospectra/runs_cnn'  #@param {type:"string"}
#@markdown GeoTIFF por célula das tasks do GEE (exportar_tiles_drive.py) e pedaços cortados
TILES_GEE_DIR = '/content/drive/MyDrive/tiles_cnn_gee'  #@param {type:"string"}
TILES_DIR = '/content/drive/MyDrive/prospectra/tiles_cnn'  #@param {type:"string"}
#@markdown Índice dos patches (pontos de fosfato para conferir o mapa)
PATCHES_DIR = '/content/drive/MyDrive/prospectra/patches_cnn'  #@param {type:"string"}

SCRIPTS = Path(SCRIPTS_DIR)
for f in ('inferir_cnn.py', 'treinar_cnn.py', 'exportar_patches.py', 'baixar_tiles.py', 'cortar_tiles_drive.py'):
    assert (SCRIPTS / f).exists(), f'falta {SCRIPTS / f} — suba o .py para essa pasta do Drive'
Path(TILES_DIR).mkdir(parents=True, exist_ok=True)
os.environ['TILES_DIR'] = TILES_DIR
os.environ['RUNS_DIR'] = RUNS_DIR
os.environ['PATCHES_DIR'] = PATCHES_DIR
sys.path.insert(0, str(SCRIPTS))
os.chdir(SCRIPTS)
print('código :', SCRIPTS)
print('rodadas:', RUNS_DIR)
print('tiles  :', TILES_GEE_DIR, '→', TILES_DIR)

In [ ]:
# [3] pacotes (rasterio e timm não vêm no Colab)
#@markdown Padrão: o torch do Colab (o mesmo do treino). Só se a célula 4 falhar: fixe o 2.8 (build cu128:
#@markdown roda em T4, L4, A100, H100 e nas Blackwell — o cu126 não tem kernel para elas) e
#@markdown **reinicie a sessão** depois desta célula (só na primeira vez)
FIXAR_TORCH = False  #@param {type:"boolean"}
if FIXAR_TORCH:
    %pip install -q torch==2.8.0 torchvision==0.23.0 --index-url https://download.pytorch.org/whl/cu128
%pip install -q timm rasterio

In [ ]:
# [4] GPU — confere se o torch instalado tem kernel para esta placa
import torch
print('torch', torch.__version__, '| CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    cap = 'sm_%d%d' % torch.cuda.get_device_capability(0)
    print(torch.cuda.get_device_name(0), f'{torch.cuda.get_device_properties(0).total_memory / 1e9:.0f} GB', '·', cap)
    archs = torch.cuda.get_arch_list()
    if cap not in archs:
        print(f'ATENÇÃO: este torch só tem kernels para {archs} — nada roda na GPU ({cap}). '
              'Reinstale na célula 3 (build cu128) e reinicie a sessão.')
    x = torch.randn(8, 16, 64, 64, device='cuda')
    torch.nn.functional.conv2d(x, torch.randn(32, 16, 3, 3, device='cuda'))
    torch.cuda.synchronize()
    print('teste de convolução na GPU: ok')
else:
    print('Sem GPU: Ambiente de execução → Alterar o tipo → GPU')


`[5]`

## Cortar os canais em pedaços (uma vez por modelo)

Os GeoTIFF por célula das tasks do GEE (`tiles_cnn_gee/canais_<modelo>_<LxCy>*.tif`)
viram os 6 × 6 pedaços com borda de 32 px (meio patch de 64: o contexto da janela na beira do pedaço) que o `inferir_cnn.py --tiles` lê, em
`tiles_cnn/canais_<modelo>/<LxCy>/`. Retomável: pedaço já gravado é pulado.
Depois de cortar, os `.tif` de `tiles_cnn_gee/` podem ser apagados (~18 GB por modelo).

In [ ]:
# [5] cortar os tiles de um modelo
import cortar_tiles_drive as ctd
MODELO_TILES = 'A'  #@param ["A", "B1"]
CELULAS_CORTE = ''  #@param {type:"string"}
ctd.main(['--modelo', MODELO_TILES, '--origem', TILES_GEE_DIR, '--destino', TILES_DIR,
          '--celulas', CELULAS_CORTE])

for m in ('A', 'B1'):
    raiz = Path(TILES_DIR) / f'canais_{m}'
    n = len(list(raiz.glob('L*C*/*_r*c*.tif'))) if raiz.exists() else 0
    print(f'canais_{m}: {n} pedaços (área toda = 900)')

In [ ]:
# [6a] ranking das rodadas — escolha aqui a que vai para o mapa (célula 6)
#@markdown Ordem da validação cruzada (ap_ponto fora do fold; célula 8 do cnn_colab). Se a
#@markdown célula 11 do cnn_colab já rodou, entram também as AUC fosfato × fundo de validação e teste.
ranking = Path(RUNS_DIR) / 'ranking_rodadas.csv'
assert ranking.exists(), f'{ranking} não existe — rode a célula 8 do cnn_colab.ipynb'
rk = pd.read_csv(ranking)
rk['modelo_mapa'] = ['final.pt (_teste)' if t else
                     ('média dos fold*.pt' if any((Path(RUNS_DIR) / r).glob('fold*.pt')) else '— sem .pt')
                     for r, t in zip(rk.rodada, rk.tem_teste)]
auc = Path(RUNS_DIR) / 'comparacao_presencas_auc.csv'
if auc.exists():
    a = pd.read_csv(auc)
    a['base'] = a.rodada.str.replace(r'_teste$', '', regex=True)
    val = a[a.grupo == 'validação'].groupby('base').auc_cnn.first().rename('auc_validacao')
    te = (a[a.grupo == 'teste'].sort_values('rodada', key=lambda s: ~s.str.endswith('_teste'))
          .groupby('base').auc_cnn.first().rename('auc_teste'))      # prefere a da rodada _teste
    rk = rk.join(val, on='rodada').join(te, on='rodada')
colunas = [c for c in ['posicao', 'rodada', 'ap_ponto_oof', 'auc_ponto_oof', 'boyce_oof', 'top10_oof',
                       'auc_validacao', 'auc_teste', 'modelo_mapa'] if c in rk]
print(rk[colunas].round(3).to_string(index=False))
print("\nNa célula 6: RODADA = 'melhor' (posição 1), a posição (ex.: '3') ou o nome da rodada.")


In [ ]:
# [6] escolher a rodada
#@markdown **melhor** = posição 1 do ranking (célula 6a); ou a posição (ex.: 3); ou o nome da
#@markdown rodada. Usa o final.pt da `<rodada>_teste` se existir; senão, a média dos fold*.pt
RODADA = 'melhor'  #@param {type:"string"}

escolha = str(RODADA).strip()
if escolha == 'melhor' or escolha.isdigit():
    rk = pd.read_csv(Path(RUNS_DIR) / 'ranking_rodadas.csv')
    pos = 1 if escolha == 'melhor' else int(escolha)
    assert pos in set(rk.posicao), f'posição {pos} fora do ranking (1–{len(rk)})'
    escolha = rk.loc[rk.posicao == pos, 'rodada'].iloc[0]
if not escolha.endswith('_teste') and (Path(RUNS_DIR) / f'{escolha}_teste' / 'final.pt').exists():
    escolha = f'{escolha}_teste'

RUN = Path(RUNS_DIR) / escolha
assert any(RUN.glob('*.pt')), f'{RUN} sem final.pt nem fold*.pt'
cfg = json.loads((RUN / 'config.json').read_text())
MODELO, PATCH = cfg['modelo'], cfg['patch']
TILES_MODELO = Path(TILES_DIR) / f'canais_{MODELO}'
n = len(list(TILES_MODELO.glob('L*C*/*_r*c*.tif'))) if TILES_MODELO.exists() else 0
print(f'usada: {RUN.name} · modelo {MODELO} · {cfg["backbone"]} · janela {PATCH} × {PATCH} px · '
      f'{"final.pt" if (RUN / "final.pt").exists() else "média dos fold*.pt"}')
print(f'pedaços {TILES_MODELO}: {n}' + ('' if n else f'  ← rode a célula 5 com MODELO_TILES = "{MODELO}"'))


`[7]`

## Mapa por janela deslizante

**Janela = o patch do treino** (64 × 64 px para os backbones; lido do
`config.json` da rodada, nada fixo aqui). Para cada pixel, a janela 64 × 64
centrada nele vai para a rede e o p sai nesse pixel. Por isso cada pixel precisa
de 32 px de vizinhos de cada lado (meio patch) — **os 32 px são a borda, não o
tamanho da janela**.

Sem borda, perderíamos as 32 primeiras e últimas linhas e colunas de cada pedaço.
Como os pedaços foram exportados com 32 px **a mais** de cada lado (vindos do
pedaço vizinho), toda a célula é prevista, sem faixas vazias entre pedaços nem
entre células. Só fica sem p o que não tem dado: a beira de fora da área do
mosaico e os buracos (pixel central sem dado → 0 no mapa).

- **Passo 3** (90 m): um p a cada 3 pixels — 9× mais rápido, para testar.
- **Passo 1** (30 m): um p por pixel, o mapa final.

Saída em `SAIDA/<rodada>/mapas_passo<k>/` (padrão: `runs_cnn/<rodada>/mapas_passo<k>/`): `pedacos/<LxCy>/p_<LxCy>_r<i>c<j>_<rodada>.tif` e, com os
36 pedaços prontos, `p_<LxCy>_<rodada>.tif` (UInt16, p × 10000, 0 = sem dado — a mesma
convenção do `favorabilidade_ml`). Retomável: pedaço já previsto é pulado.
Faltou memória na GPU: baixe `LOTE` ou suba `SUB`.

In [ ]:
# [7] inferir
import inferir_cnn as ic
PASSO = 3  #@param [3, 1] {type:"raw"}
#@markdown Vazio = todas as 25 células; para testar: L2C5
CELULAS = 'L2C5'  #@param {type:"string"}
LOTE = 1024  #@param [256, 512, 1024, 2048] {type:"raw"}
SUB = 1  #@param [1, 2, 4] {type:"raw"}
#@markdown Pasta de saída: vazio = dentro da rodada (runs_cnn/<rodada>/mapas_passo<k>); outra pasta
#@markdown (ex.: /content/drive/MyDrive/prospectra/mapas_cnn) → <pasta>/<rodada>/mapas_passo<k>
SAIDA = ''  #@param {type:"string"}

MAPAS = (Path(SAIDA) / RUN.name if SAIDA.strip() else RUN) / f'mapas_passo{PASSO}'
print('saída:', MAPAS)

ic.main(['--run', str(RUN), '--tiles', str(TILES_MODELO), '--passo', str(PASSO),
         '--celulas', CELULAS, '--lote', str(LOTE), '--sub', str(SUB), '--saida', str(MAPAS)])

In [ ]:
# [8] ver o mapa e o p nos pontos de fosfato
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import rasterio
from rasterio.enums import Resampling

#@markdown Redução para desenhar (o arquivo não muda)
REDUCAO = 8  #@param {type:"integer"}
arquivos = sorted(MAPAS.glob(f'p_L*C*_{RUN.name}.tif'))
assert arquivos, f'nenhum p_<LxCy>.tif em {MAPAS} — a célula 7 terminou alguma célula? (nomes antigos sem o sufixo da rodada: rode a 7 de novo)'

fig, ax = plt.subplots(figsize=(10, 10))
resumo = []
for arq in arquivos:
    with rasterio.open(arq) as d:
        v = d.read(1, out_shape=(max(1, d.height // REDUCAO), max(1, d.width // REDUCAO)),
                   resampling=Resampling.average if REDUCAO > 1 else Resampling.nearest)
        b = d.bounds
        tudo = d.read(1)
    p = np.where(v > 0, v / 10000, np.nan)
    im = ax.imshow(p, extent=(b.left, b.right, b.bottom, b.top), cmap='magma', vmin=0, vmax=1,
                   interpolation='nearest')
    pt = tudo[tudo > 0] / 10000
    resumo.append({'celula': arq.name.split('_')[1], 'p_mediano': np.median(pt), 'p>=0,5 (%)': 100 * np.mean(pt >= 0.5)})
ax.autoscale()

# pontos de fosfato (sem jitter): treino × teste
pontos = pd.DataFrame()
indice = Path(PATCHES_DIR) / 'indice_patches.csv'
if indice.exists():
    idx = pd.read_csv(indice)
    pontos = idx[idx.origem_amostra.str.startswith('ocorrencia') & (idx.jitter == 0)].copy()
    for origem, mk, cor in [('ocorrencia_treino', 'o', 'white'), ('ocorrencia_teste', 'x', 'cyan')]:
        s = pontos[pontos.origem_amostra == origem]
        ax.scatter(s.lon, s.lat, marker=mk, s=18, c=cor if mk == 'x' else 'none',
                   edgecolors=cor if mk == 'o' else None, linewidths=1, label=f'{origem} ({len(s)})')
    ax.legend(loc='lower right', fontsize=8)
plt.colorbar(im, ax=ax, shrink=0.7, label='p CNN')
ax.set_title(f'{RUN.name} · passo {PASSO} ({PASSO * 30} m)')
plt.show()
print(pd.DataFrame(resumo).round(3).to_string(index=False))

# p do mapa em cada ponto e o percentil desse p na própria célula (alto = o mapa destaca o ponto)
if len(pontos):
    linhas = []
    for arq in arquivos:
        cel = arq.name.split('_')[1]
        s = pontos[pontos.celula == cel]
        if not len(s):
            continue
        with rasterio.open(arq) as d:
            tudo = d.read(1)
            valores = np.array([x[0] for x in d.sample(zip(s.lon, s.lat))])
        fundo = np.sort(tudo[tudo > 0])
        for (_, r), v in zip(s.iterrows(), valores):
            if v > 0:
                linhas.append({'celula': cel, 'origem': r.origem_amostra, 'p_cnn': v / 10000,
                               'percentil': 100 * np.searchsorted(fundo, v) / len(fundo)})
    if linhas:
        t = pd.DataFrame(linhas)
        print('\np nos pontos de fosfato (percentil = posição do p entre todos os pixels da célula):')
        print(t.groupby('origem').agg(n=('p_cnn', 'size'), p_mediano=('p_cnn', 'median'),
                                      percentil_mediano=('percentil', 'median'),
                                      no_top10=('percentil', lambda x: 100 * np.mean(x >= 90))).round(2).to_string())

`[9]`

## Subir ao GEE (opcional)

Envia os `p_<LxCy>_<rodada>.tif` de `mapas_passo<k>/` ao bucket e cria as imagens na
coleção `cnn_favorabilidade` (mesmo formato do `favorabilidade_ml`, para comparar
com RF/GTB no `visualizar_favorabilidade_ml.js`). Pede login do Earth Engine e do
Google Cloud; a conta precisa escrever no bucket e em `projects/mapbiomas-arida/mine/`.
Se preferir, baixe a pasta `mapas_passo<k>/` e rode o mesmo script na máquina local.

In [ ]:
# [9] subir ao GEE
SUBIR = False  #@param {type:"boolean"}
PROJETO_EE = 'mapbiomas-brazil'  #@param {type:"string"}
if SUBIR:
    from google.colab import auth
    import ee
    auth.authenticate_user()                 # credenciais do Cloud Storage (ADC)
    ee.Authenticate()
    ee.Initialize(project=PROJETO_EE)
    import subir_predicao_gee as sp
    sp.main(['--run', str(RUN), '--passo', str(PASSO), '--mapas', str(MAPAS), '--projeto', PROJETO_EE])